In [ ]:
# Importe
from pathlib import Path
import json
import platform
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
try:
    import pm4py
except ImportError as e:
    raise ImportError('PM4Py ist nicht installiert. Bitte in der aktivierten .venv ausführen: pip install -U pm4py') from e
print('Imports OK')
print('Python:', platform.python_version())
print('pandas:', pd.__version__)


In [ ]:
# Pfade und Einstellungen
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data_raw'
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'label_robustness_feature_availability'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
REUSE_DESCRIPTIVE_CASE_FEATURES_IF_AVAILABLE = True
BUILD_PREFIX_FEATURES = True
PREFIX_K_LIST = [5, 10, 20]
PREFIX_DAY_LIST = [0, 30, 90, 180]
MIN_CASES_PER_GROUP_FOR_DEPT_THRESHOLDS = 100
PRIMARY_LABEL = 'label_scd_p90_or_global'
print('Project root:', PROJECT_ROOT)
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Datensatzpfad suchen
if not LOG_PATH.exists():
    candidates = sorted(DATA_RAW.glob('*.xes*')) + sorted(DATA_RAW.glob('**/*.xes*'))
    print('Gefundene XES-Kandidaten:')
    for c in candidates[:20]:
        print('-', c)
    if candidates:
        LOG_PATH = candidates[0]
        print('Nutze automatisch:', LOG_PATH)
    else:
        raise FileNotFoundError(f'Keine XES/XES.GZ-Datei in {DATA_RAW} gefunden.')
print('Log path:', LOG_PATH)
print('Exists:', LOG_PATH.exists())


In [ ]:
# Hilfsfunktionen
created_tables = []
created_figures = []

def save_csv(obj, filename, index=True):
    path = TABLE_DIR / filename
    if isinstance(obj, pd.Series):
        obj.to_frame().to_csv(path, index=index, encoding='utf-8-sig')
    else:
        obj.to_csv(path, index=index, encoding='utf-8-sig')
    created_tables.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    created_tables.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(path)
    return path

def safe_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def sanitize_name(x):
    return str(x).replace(':', '_').replace('/', '_').replace(' ', '_').replace('-', '_')

def cramers_v_from_crosstab(tab):
    arr = tab.values.astype(float)
    n = arr.sum()
    if n == 0 or arr.shape[0] < 2 or arr.shape[1] < 2:
        return np.nan
    row_sum = arr.sum(axis=1, keepdims=True)
    col_sum = arr.sum(axis=0, keepdims=True)
    expected = row_sum @ col_sum / n
    with np.errstate(divide='ignore', invalid='ignore'):
        chi2 = np.nansum((arr - expected) ** 2 / expected)
    phi2 = chi2 / n
    denom = min(arr.shape[0] - 1, arr.shape[1] - 1)
    if denom <= 0:
        return np.nan
    return float(np.sqrt(phi2 / denom))

def jaccard(a, b):
    a = pd.Series(a).fillna(False).astype(bool)
    b = pd.Series(b).fillna(False).astype(bool)
    union = (a | b).sum()
    if union == 0:
        return np.nan
    return float((a & b).sum() / union)

def pair_overlap_metrics(df_labels, label_a, label_b):
    a = df_labels[label_a].fillna(False).astype(bool)
    b = df_labels[label_b].fillna(False).astype(bool)
    n = len(df_labels)
    both = int((a & b).sum())
    only_a = int((a & ~b).sum())
    only_b = int((~a & b).sum())
    neither = int((~a & ~b).sum())
    return {'label_a': label_a, 'label_b': label_b, 'both_true': both, 'only_a': only_a, 'only_b': only_b, 'neither': neither, 'jaccard': round(jaccard(a, b), 4), 'pct_a_also_b': round(float(b[a].mean() * 100), 4) if a.sum() else np.nan, 'pct_b_also_a': round(float(a[b].mean() * 100), 4) if b.sum() else np.nan, 'n_cases': n}
print('Helper OK')


In [ ]:
# Log laden
print('Lade BPIC-Log. Das kann einige Zeit dauern ...')
raw_log = pm4py.read_xes(str(LOG_PATH))
if isinstance(raw_log, pd.DataFrame):
    event_df = raw_log.copy()
else:
    event_df = pm4py.convert_to_dataframe(raw_log)
print('event_df shape:', event_df.shape)
event_df.head()


In [ ]:
# Kernspalten vorbereiten
CASE_COL = safe_first_existing(event_df.columns, ['case:concept:name', 'case_id', 'case'])
ACTIVITY_COL = safe_first_existing(event_df.columns, ['concept:name', 'activity'])
RAW_ACTIVITY_COL = 'activity' if 'activity' in event_df.columns else ACTIVITY_COL
TIME_COL = safe_first_existing(event_df.columns, ['time:timestamp', 'timestamp', 'time'])
ORDER_COL = safe_first_existing(event_df.columns, ['identity:id', 'eventid', 'event_id'])
if CASE_COL is None or ACTIVITY_COL is None or TIME_COL is None:
    raise RuntimeError(f'Kernspalten fehlen: CASE_COL={CASE_COL}, ACTIVITY_COL={ACTIVITY_COL}, TIME_COL={TIME_COL}')
event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors='coerce', utc=True)
if ORDER_COL is None:
    event_df['_technical_order'] = np.arange(len(event_df))
    ORDER_COL = '_technical_order'
if all((c in event_df.columns for c in ['doctype', 'subprocess', RAW_ACTIVITY_COL])):
    event_df['combined_activity'] = event_df['doctype'].astype(str) + ' | ' + event_df['subprocess'].astype(str) + ' | ' + event_df[RAW_ACTIVITY_COL].astype(str)
else:
    event_df['combined_activity'] = event_df[ACTIVITY_COL].astype(str)
event_df = event_df.sort_values([CASE_COL, TIME_COL, ORDER_COL], kind='mergesort').reset_index(drop=True)
event_df['event_position'] = event_df.groupby(CASE_COL).cumcount() + 1
case_start_series = event_df.groupby(CASE_COL)[TIME_COL].transform('min')
event_df['days_since_case_start'] = (event_df[TIME_COL] - case_start_series).dt.total_seconds() / (3600 * 24)
basic_info = {'events': int(len(event_df)), 'cases': int(event_df[CASE_COL].nunique()), 'activities': int(event_df[ACTIVITY_COL].nunique()), 'combined_activities': int(event_df['combined_activity'].nunique()), 'columns': int(event_df.shape[1]), 'timestamp_min': str(event_df[TIME_COL].min()), 'timestamp_max': str(event_df[TIME_COL].max()), 'case_col': CASE_COL, 'activity_col': ACTIVITY_COL, 'raw_activity_col': RAW_ACTIVITY_COL, 'time_col': TIME_COL, 'order_col': ORDER_COL}
save_json(basic_info, '00_basic_info.json')
pd.DataFrame([basic_info])


In [ ]:
# Fallmerkmale laden
previous_case_features_path = PROJECT_ROOT / 'outputs' / 'descriptive_process_analysis' / 'tables' / '35_case_level_features_descriptive_analysis.csv'
use_previous_case_features = False
if REUSE_DESCRIPTIVE_CASE_FEATURES_IF_AVAILABLE and previous_case_features_path.exists():
    try:
        case_df = pd.read_csv(previous_case_features_path)
        if CASE_COL in case_df.columns or 'case:concept:name' in case_df.columns:
            use_previous_case_features = True
            print('Nutze vorhandene Case-Level-Features aus:', previous_case_features_path)
        else:
            print('Vorhandene Case-Level-Datei gefunden, aber Case-ID fehlt. Rekonstruiere neu.')
    except Exception as e:
        print('Konnte vorhandene Case-Level-Datei nicht laden. Rekonstruiere neu.', e)
if not use_previous_case_features:
    print('Baue Case-Level-Features aus Event Log ...')
    case_df = event_df.groupby(CASE_COL).agg(event_count=(ACTIVITY_COL, 'size'), case_start=(TIME_COL, 'min'), case_end=(TIME_COL, 'max'), n_raw_activities=(ACTIVITY_COL, 'nunique'), n_combined_activities=('combined_activity', 'nunique')).reset_index()
    if 'docid' in event_df.columns:
        case_df = case_df.merge(event_df.groupby(CASE_COL)['docid'].nunique().rename('n_docs'), on=CASE_COL, how='left')
    if 'doctype' in event_df.columns:
        case_df = case_df.merge(event_df.groupby(CASE_COL)['doctype'].nunique().rename('n_doctypes'), on=CASE_COL, how='left')
    if 'subprocess' in event_df.columns:
        case_df = case_df.merge(event_df.groupby(CASE_COL)['subprocess'].nunique().rename('n_subprocesses'), on=CASE_COL, how='left')
    if 'org:resource' in event_df.columns:
        case_df = case_df.merge(event_df.groupby(CASE_COL)['org:resource'].nunique().rename('n_resources'), on=CASE_COL, how='left')
    case_df['duration_days'] = (case_df['case_end'] - case_df['case_start']).dt.total_seconds() / (3600 * 24)
    case_df['start_calendar_year'] = pd.to_datetime(case_df['case_start'], utc=True).dt.year
    case_df['end_calendar_year'] = pd.to_datetime(case_df['case_end'], utc=True).dt.year
    case_cols = [c for c in event_df.columns if c.startswith('case:') and c != CASE_COL]
    if case_cols:
        static_case_attrs = event_df.groupby(CASE_COL)[case_cols].first().reset_index()
        case_df = case_df.merge(static_case_attrs, on=CASE_COL, how='left')
    for source_col, prefix in [(ACTIVITY_COL, 'act'), ('subprocess', 'subprocess'), ('doctype', 'doctype')]:
        if source_col in event_df.columns:
            counts = event_df.groupby([CASE_COL, source_col]).size().unstack(fill_value=0)
            counts.columns = [f'{prefix}__{sanitize_name(c)}' for c in counts.columns]
            case_df = case_df.merge(counts.reset_index(), on=CASE_COL, how='left')
    act_counts = event_df.groupby([CASE_COL, ACTIVITY_COL]).size().reset_index(name='count')
    raw_rep = act_counts[act_counts['count'] > 1]
    raw_rework = raw_rep.groupby(CASE_COL).agg(repeated_activity_types=(ACTIVITY_COL, 'nunique'), repeated_activity_total_extra=('count', lambda x: int((x - 1).sum())), max_repeat_single_activity=('count', 'max')).reset_index()
    case_df = case_df.merge(raw_rework, on=CASE_COL, how='left')
    comb_counts = event_df.groupby([CASE_COL, 'combined_activity']).size().reset_index(name='count')
    comb_rep = comb_counts[comb_counts['count'] > 1]
    comb_rework = comb_rep.groupby(CASE_COL).agg(repeated_combined_activity_types=('combined_activity', 'nunique'), repeated_combined_activity_total_extra=('count', lambda x: int((x - 1).sum())), max_repeat_single_combined_activity=('count', 'max')).reset_index()
    case_df = case_df.merge(comb_rework, on=CASE_COL, how='left')
if CASE_COL not in case_df.columns and 'case:concept:name' in case_df.columns:
    CASE_COL = 'case:concept:name'
for c in ['repeated_activity_types', 'repeated_activity_total_extra', 'max_repeat_single_activity', 'repeated_combined_activity_types', 'repeated_combined_activity_total_extra', 'max_repeat_single_combined_activity', 'n_docs', 'n_doctypes', 'n_subprocesses', 'n_resources']:
    if c in case_df.columns:
        case_df[c] = pd.to_numeric(case_df[c], errors='coerce').fillna(0)
if 'combined_rework_extra' not in case_df.columns:
    if 'repeated_combined_activity_total_extra' in case_df.columns:
        case_df['combined_rework_extra'] = pd.to_numeric(case_df['repeated_combined_activity_total_extra'], errors='coerce').fillna(0)
    else:
        case_df['combined_rework_extra'] = 0
for c in ['event_count', 'duration_days', 'combined_rework_extra']:
    case_df[c] = pd.to_numeric(case_df[c], errors='coerce')
CASE_YEAR_COL = safe_first_existing(case_df.columns, ['case:year', 'case_year', 'start_calendar_year'])
CASE_DEPT_COL = safe_first_existing(case_df.columns, ['case:department', 'department', 'org:resource'])
if CASE_YEAR_COL is None:
    case_df['_case_year_fallback'] = pd.to_datetime(case_df['case_start'], utc=True, errors='coerce').dt.year
    CASE_YEAR_COL = '_case_year_fallback'
print('case_df shape:', case_df.shape)
print('CASE_YEAR_COL:', CASE_YEAR_COL)
print('CASE_DEPT_COL:', CASE_DEPT_COL)
case_df.head()


In [ ]:
# Inspection Merkmale
def contains_ci(series, pattern):
    return series.astype(str).str.lower().str.contains(pattern, regex=True, na=False)
inspection_mask = pd.Series(False, index=event_df.index)
if 'doctype' in event_df.columns:
    inspection_mask |= contains_ci(event_df['doctype'], 'inspection')
if 'subprocess' in event_df.columns:
    inspection_mask |= contains_ci(event_df['subprocess'], 'on.?site|remote|inspection')
if 'combined_activity' in event_df.columns:
    inspection_mask |= contains_ci(event_df['combined_activity'], 'inspection|on.?site|remote')
event_df['is_inspection_context'] = inspection_mask.astype(bool)
inspection_by_case = event_df.groupby(CASE_COL)['is_inspection_context'].agg(inspection_event_count='sum', has_inspection_context='max').reset_index()
case_df = case_df.merge(inspection_by_case, on=CASE_COL, how='left')
case_df['inspection_event_count'] = case_df['inspection_event_count'].fillna(0).astype(int)
case_df['has_inspection_context'] = case_df['has_inspection_context'].fillna(False).astype(bool)
non_inspection_events = event_df[~event_df['is_inspection_context']].copy()
noninsp_event_count = non_inspection_events.groupby(CASE_COL).size().rename('event_count_no_inspection').reset_index()
case_df = case_df.merge(noninsp_event_count, on=CASE_COL, how='left')
case_df['event_count_no_inspection'] = case_df['event_count_no_inspection'].fillna(0).astype(int)
if len(non_inspection_events) > 0:
    noninsp_comb_counts = non_inspection_events.groupby([CASE_COL, 'combined_activity']).size().reset_index(name='count')
    noninsp_comb_rep = noninsp_comb_counts[noninsp_comb_counts['count'] > 1]
    noninsp_rework = noninsp_comb_rep.groupby(CASE_COL).agg(combined_rework_extra_no_inspection=('count', lambda x: int((x - 1).sum())), repeated_combined_types_no_inspection=('combined_activity', 'nunique')).reset_index()
    case_df = case_df.merge(noninsp_rework, on=CASE_COL, how='left')
else:
    case_df['combined_rework_extra_no_inspection'] = 0
    case_df['repeated_combined_types_no_inspection'] = 0
case_df['combined_rework_extra_no_inspection'] = case_df.get('combined_rework_extra_no_inspection', 0)
case_df['combined_rework_extra_no_inspection'] = pd.to_numeric(case_df['combined_rework_extra_no_inspection'], errors='coerce').fillna(0)
case_df['repeated_combined_types_no_inspection'] = pd.to_numeric(case_df.get('repeated_combined_types_no_inspection', 0), errors='coerce').fillna(0)
inspection_summary = pd.DataFrame({'metric': ['events_inspection_context', 'pct_events_inspection_context', 'cases_with_inspection_context', 'pct_cases_with_inspection_context'], 'value': [int(event_df['is_inspection_context'].sum()), round(float(event_df['is_inspection_context'].mean() * 100), 4), int(case_df['has_inspection_context'].sum()), round(float(case_df['has_inspection_context'].mean() * 100), 4)]})
save_csv(inspection_summary, '01_inspection_context_summary.csv', index=False)
inspection_summary


In [ ]:
# Labelvarianten
def quantile_threshold(series, q):
    return float(pd.to_numeric(series, errors='coerce').quantile(q))

def numeric_series(df_, col, default=0):
    if col in df_.columns:
        return pd.to_numeric(df_[col], errors='coerce').fillna(default)
    return pd.Series(default, index=df_.index)

def bool_series(df_, col, default=False):
    if col in df_.columns:
        return df_[col].fillna(default).astype(bool)
    return pd.Series(default, index=df_.index)

def ensure_numeric_col(df_, col, default=0):
    if col not in df_.columns:
        df_[col] = default
    df_[col] = pd.to_numeric(df_[col], errors='coerce').fillna(default)
    return df_
required_numeric_cols = ['event_count', 'combined_rework_extra', 'duration_days', 'event_count_no_inspection', 'combined_rework_extra_no_inspection']
for col in required_numeric_cols:
    case_df = ensure_numeric_col(case_df, col, default=0)
if 'CASE_YEAR_COL' not in globals():
    CASE_YEAR_COL = None
if CASE_YEAR_COL is None or CASE_YEAR_COL not in case_df.columns:
    possible_year_cols = [c for c in case_df.columns if any((x in c.lower() for x in ['year', 'jahr']))]
    if len(possible_year_cols) > 0:
        CASE_YEAR_COL = possible_year_cols[0]
        print(f'CASE_YEAR_COL automatisch gesetzt auf: {CASE_YEAR_COL}')
    elif 'case_start_time' in case_df.columns:
        CASE_YEAR_COL = '_case_year_for_norm'
        case_df[CASE_YEAR_COL] = pd.to_datetime(case_df['case_start_time'], errors='coerce').dt.year.astype('Int64').astype(str)
        print('CASE_YEAR_COL aus case_start_time erzeugt.')
    elif 'start_time' in case_df.columns:
        CASE_YEAR_COL = '_case_year_for_norm'
        case_df[CASE_YEAR_COL] = pd.to_datetime(case_df['start_time'], errors='coerce').dt.year.astype('Int64').astype(str)
        print('CASE_YEAR_COL aus start_time erzeugt.')
    elif TIME_COL in event_df.columns:
        tmp_year = event_df.groupby(CASE_COL)[TIME_COL].min()
        tmp_year = pd.to_datetime(tmp_year, errors='coerce').dt.year.astype('Int64').astype(str)
        tmp_year_df = tmp_year.rename('_case_year_for_norm').reset_index()
        case_df = case_df.merge(tmp_year_df, on=CASE_COL, how='left')
        CASE_YEAR_COL = '_case_year_for_norm'
        print('CASE_YEAR_COL aus erstem Event-Timestamp erzeugt.')
    else:
        CASE_YEAR_COL = '_case_year_for_norm'
        case_df[CASE_YEAR_COL] = 'unknown'
        print("Warnung: Keine Jahr-Spalte gefunden. CASE_YEAR_COL auf 'unknown' gesetzt.")
case_df[CASE_YEAR_COL] = case_df[CASE_YEAR_COL].astype(str).fillna('unknown')
thresholds = {'event_count_p90_global': quantile_threshold(case_df['event_count'], 0.9), 'event_count_p95_global': quantile_threshold(case_df['event_count'], 0.95), 'combined_rework_p90_global': quantile_threshold(case_df['combined_rework_extra'], 0.9), 'combined_rework_p95_global': quantile_threshold(case_df['combined_rework_extra'], 0.95), 'duration_p90_global': quantile_threshold(case_df['duration_days'], 0.9), 'duration_p95_global': quantile_threshold(case_df['duration_days'], 0.95), 'event_count_no_inspection_p90_global': quantile_threshold(case_df['event_count_no_inspection'], 0.9), 'combined_rework_no_inspection_p90_global': quantile_threshold(case_df['combined_rework_extra_no_inspection'], 0.9)}
case_df['label_scd_p90_or_global'] = (case_df['event_count'] >= thresholds['event_count_p90_global']) | (case_df['combined_rework_extra'] >= thresholds['combined_rework_p90_global'])
case_df['label_scd_p95_or_global'] = (case_df['event_count'] >= thresholds['event_count_p95_global']) | (case_df['combined_rework_extra'] >= thresholds['combined_rework_p95_global'])
case_df['label_scd_p90_and_global'] = (case_df['event_count'] >= thresholds['event_count_p90_global']) & (case_df['combined_rework_extra'] >= thresholds['combined_rework_p90_global'])
case_df['label_scd_p95_and_global'] = (case_df['event_count'] >= thresholds['event_count_p95_global']) & (case_df['combined_rework_extra'] >= thresholds['combined_rework_p95_global'])
case_df['label_temporal_duration_p90_global'] = case_df['duration_days'] >= thresholds['duration_p90_global']
case_df['label_temporal_duration_p95_global'] = case_df['duration_days'] >= thresholds['duration_p95_global']
case_df['label_scd_p90_or_no_inspection_global'] = (case_df['event_count_no_inspection'] >= thresholds['event_count_no_inspection_p90_global']) | (case_df['combined_rework_extra_no_inspection'] >= thresholds['combined_rework_no_inspection_p90_global'])
metrics_for_year_norm = ['event_count', 'combined_rework_extra', 'duration_days', 'event_count_no_inspection', 'combined_rework_extra_no_inspection']
for metric in metrics_for_year_norm:
    for qname, q in [('p90', 0.9), ('p95', 0.95)]:
        t_col = f'_thr_{metric}_{qname}_by_year'
        case_df[t_col] = case_df.groupby(CASE_YEAR_COL)[metric].transform(lambda s: pd.to_numeric(s, errors='coerce').quantile(q))
case_df['label_scd_p90_or_yearnorm'] = (case_df['event_count'] >= case_df['_thr_event_count_p90_by_year']) | (case_df['combined_rework_extra'] >= case_df['_thr_combined_rework_extra_p90_by_year'])
case_df['label_scd_p95_or_yearnorm'] = (case_df['event_count'] >= case_df['_thr_event_count_p95_by_year']) | (case_df['combined_rework_extra'] >= case_df['_thr_combined_rework_extra_p95_by_year'])
case_df['label_scd_p90_and_yearnorm'] = (case_df['event_count'] >= case_df['_thr_event_count_p90_by_year']) & (case_df['combined_rework_extra'] >= case_df['_thr_combined_rework_extra_p90_by_year'])
case_df['label_temporal_duration_p90_yearnorm'] = case_df['duration_days'] >= case_df['_thr_duration_days_p90_by_year']
case_df['label_scd_p90_or_no_inspection_yearnorm'] = (case_df['event_count_no_inspection'] >= case_df['_thr_event_count_no_inspection_p90_by_year']) | (case_df['combined_rework_extra_no_inspection'] >= case_df['_thr_combined_rework_extra_no_inspection_p90_by_year'])
if 'CASE_DEPT_COL' not in globals():
    CASE_DEPT_COL = None
if 'MIN_CASES_PER_GROUP_FOR_DEPT_THRESHOLDS' not in globals():
    MIN_CASES_PER_GROUP_FOR_DEPT_THRESHOLDS = 100
if CASE_DEPT_COL is not None and CASE_DEPT_COL in case_df.columns:
    dept_as_str = case_df[CASE_DEPT_COL].astype(str).fillna('missing')
    dept_counts = dept_as_str.value_counts(dropna=False)
    eligible_depts = set(dept_counts[dept_counts >= MIN_CASES_PER_GROUP_FOR_DEPT_THRESHOLDS].index.astype(str))
    case_df['_dept_for_threshold'] = dept_as_str.where(dept_as_str.isin(eligible_depts), '__small_or_missing__')
    for metric in ['event_count', 'combined_rework_extra']:
        case_df[f'_thr_{metric}_p90_by_dept'] = case_df.groupby('_dept_for_threshold')[metric].transform(lambda s: pd.to_numeric(s, errors='coerce').quantile(0.9))
    case_df['label_scd_p90_or_deptnorm'] = (case_df['event_count'] >= case_df['_thr_event_count_p90_by_dept']) | (case_df['combined_rework_extra'] >= case_df['_thr_combined_rework_extra_p90_by_dept'])
else:
    case_df['label_scd_p90_or_deptnorm'] = False
for old_col in ['label_path_change_or_objection', 'label_path_change_or_objection_x', 'label_path_change_or_objection_y']:
    if old_col in case_df.columns:
        case_df = case_df.drop(columns=[old_col])
if 'subprocess__Change' in case_df.columns or 'subprocess__Objection' in case_df.columns:
    change_count = numeric_series(case_df, 'subprocess__Change', default=0)
    objection_count = numeric_series(case_df, 'subprocess__Objection', default=0)
    case_df['label_path_change_or_objection'] = ((change_count > 0) | (objection_count > 0)).astype(bool)
elif 'subprocess' in event_df.columns:
    by_case_sub = event_df.groupby([CASE_COL, 'subprocess']).size().unstack(fill_value=0)
    change_like = [c for c in by_case_sub.columns if 'change' in str(c).lower()]
    objection_like = [c for c in by_case_sub.columns if 'objection' in str(c).lower()]
    relevant_subprocess_cols = change_like + objection_like
    if len(relevant_subprocess_cols) > 0:
        tmp_label = by_case_sub[relevant_subprocess_cols].sum(axis=1).gt(0).rename('label_path_change_or_objection').reset_index()
    else:
        tmp_label = pd.DataFrame({CASE_COL: case_df[CASE_COL].values, 'label_path_change_or_objection': False})
    case_df = case_df.merge(tmp_label, on=CASE_COL, how='left')
    case_df['label_path_change_or_objection'] = case_df['label_path_change_or_objection'].fillna(False).astype(bool)
else:
    case_df['label_path_change_or_objection'] = False
if 'label_remove_document' not in case_df.columns:
    act_remove_cols = [c for c in case_df.columns if c.lower().startswith('act__remove_document')]
    if len(act_remove_cols) > 0:
        case_df['label_remove_document'] = case_df[act_remove_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).gt(0)
    elif 'activity' in event_df.columns:
        tmp_remove = event_df.assign(_is_remove_document=event_df['activity'].astype(str).str.lower().eq('remove document')).groupby(CASE_COL)['_is_remove_document'].any().rename('label_remove_document').reset_index()
        case_df = case_df.merge(tmp_remove, on=CASE_COL, how='left')
        case_df['label_remove_document'] = case_df['label_remove_document'].fillna(False).astype(bool)
    elif 'concept:name' in event_df.columns:
        tmp_remove = event_df.assign(_is_remove_document=event_df['concept:name'].astype(str).str.lower().eq('remove document')).groupby(CASE_COL)['_is_remove_document'].any().rename('label_remove_document').reset_index()
        case_df = case_df.merge(tmp_remove, on=CASE_COL, how='left')
        case_df['label_remove_document'] = case_df['label_remove_document'].fillna(False).astype(bool)
    else:
        case_df['label_remove_document'] = False
case_df['label_remove_document'] = case_df['label_remove_document'].fillna(False).astype(bool)
LABEL_COLS = ['label_scd_p90_or_global', 'label_scd_p95_or_global', 'label_scd_p90_and_global', 'label_scd_p95_and_global', 'label_scd_p90_or_yearnorm', 'label_scd_p95_or_yearnorm', 'label_scd_p90_and_yearnorm', 'label_scd_p90_or_deptnorm', 'label_scd_p90_or_no_inspection_global', 'label_scd_p90_or_no_inspection_yearnorm', 'label_temporal_duration_p90_global', 'label_temporal_duration_p95_global', 'label_temporal_duration_p90_yearnorm', 'label_path_change_or_objection', 'label_remove_document']
LABEL_COLS = [c for c in LABEL_COLS if c in case_df.columns]
for col in LABEL_COLS:
    if case_df[col].dtype != bool:
        case_df[col] = case_df[col].fillna(False).astype(bool)
save_json(thresholds, '02_label_thresholds.json')
print('Labels constructed:')
for col in LABEL_COLS:
    share = case_df[col].mean() * 100
    count = int(case_df[col].sum())
    print(f'- {col}: {count:,} cases ({share:.2f}%)')
case_df[[CASE_COL] + LABEL_COLS].head()


In [ ]:
# Labelprävalenz und Überschneidungen
label_prevalence_rows = []
for label in LABEL_COLS:
    s = case_df[label]
    if s.dtype != bool:
        s = s.fillna(False).astype(bool)
    cases_true = int(s.sum())
    label_prevalence_rows.append({'label': label, 'cases_true': cases_true, 'share_pct': round(cases_true / len(case_df) * 100, 4), 'suitability_prevalence_comment': 'zu selten für einfache Hauptklassifikation' if cases_true / len(case_df) < 0.03 else 'breit, aber noch fokussierbar' if cases_true / len(case_df) <= 0.2 else 'sehr breit / Risiko unscharfes Target'})
label_prevalence = pd.DataFrame(label_prevalence_rows).sort_values('share_pct', ascending=False)
save_csv(label_prevalence, '03_label_prevalence.csv', index=False)
label_prevalence


In [ ]:
# Jaccard Vergleich
pair_rows = []
for label in LABEL_COLS:
    if label == PRIMARY_LABEL:
        continue
    pair_rows.append(pair_overlap_metrics(case_df, PRIMARY_LABEL, label))
primary_overlap = pd.DataFrame(pair_rows).sort_values('jaccard', ascending=False)
save_csv(primary_overlap, '04_overlap_with_primary_label.csv', index=False)
primary_overlap


In [ ]:
# Jaccard Matrix
jaccard_matrix = pd.DataFrame(index=LABEL_COLS, columns=LABEL_COLS, dtype=float)
for a in LABEL_COLS:
    for b in LABEL_COLS:
        jaccard_matrix.loc[a, b] = jaccard(case_df[a], case_df[b])
save_csv(jaccard_matrix, '05_label_jaccard_matrix.csv')
jaccard_matrix


In [ ]:
# Stabilitätsanalyse
def label_rate_by_group(df, label, group_col, min_n=1):
    if group_col is None or group_col not in df.columns:
        return pd.DataFrame()
    tmp = df[[group_col, label]].copy()
    tmp[label] = tmp[label].fillna(False).astype(bool)
    out = tmp.groupby(group_col).agg(cases=(label, 'size'), positives=(label, 'sum'), share_positive_pct=(label, lambda s: float(s.mean() * 100))).reset_index()
    out = out[out['cases'] >= min_n].copy()
    out['label'] = label
    return out.sort_values(group_col)
stability_rows = []
for label in LABEL_COLS:
    for group_name, group_col, min_n in [('case_year', CASE_YEAR_COL, 1), ('case_department', CASE_DEPT_COL, 30 if CASE_DEPT_COL else 1), ('start_calendar_year', 'start_calendar_year', 1 if 'start_calendar_year' in case_df.columns else 999999), ('inspection_context', 'has_inspection_context', 1)]:
        if group_col is None or group_col not in case_df.columns:
            continue
        rates = label_rate_by_group(case_df, label, group_col, min_n=min_n)
        if len(rates) == 0:
            continue
        tab = pd.crosstab(case_df[group_col], case_df[label].fillna(False).astype(bool))
        stability_rows.append({'label': label, 'group': group_name, 'group_col': group_col, 'n_groups': int(rates.shape[0]), 'min_share_pct': round(float(rates['share_positive_pct'].min()), 4), 'max_share_pct': round(float(rates['share_positive_pct'].max()), 4), 'range_pp': round(float(rates['share_positive_pct'].max() - rates['share_positive_pct'].min()), 4), 'std_share_pct': round(float(rates['share_positive_pct'].std(ddof=0)), 4), 'cramers_v': round(cramers_v_from_crosstab(tab), 4)})
stability_summary = pd.DataFrame(stability_rows).sort_values(['label', 'group'])
save_csv(stability_summary, '06_label_stability_summary.csv', index=False)
stability_summary


In [ ]:
# Detailtabellen
central_labels = [PRIMARY_LABEL, 'label_scd_p95_or_global', 'label_scd_p90_or_yearnorm', 'label_scd_p90_or_no_inspection_global', 'label_temporal_duration_p90_global', 'label_path_change_or_objection']
central_labels = [c for c in central_labels if c in LABEL_COLS]
year_detail = pd.concat([label_rate_by_group(case_df, label, CASE_YEAR_COL) for label in central_labels], ignore_index=True)
save_csv(year_detail, '07_central_labels_by_case_year.csv', index=False)
if CASE_DEPT_COL is not None and CASE_DEPT_COL in case_df.columns:
    dept_detail = pd.concat([label_rate_by_group(case_df, label, CASE_DEPT_COL, min_n=30) for label in central_labels], ignore_index=True)
    save_csv(dept_detail, '08_central_labels_by_department_min30.csv', index=False)
else:
    dept_detail = pd.DataFrame()
year_detail.head(30)


In [ ]:
# Inspection Abhängigkeit
inspection_detail = []
if 'has_inspection_context' in case_df.columns:
    for label in central_labels:
        rates = label_rate_by_group(case_df, label, 'has_inspection_context')
        for _, r in rates.iterrows():
            inspection_detail.append({'label': label, 'has_inspection_context': bool(r['has_inspection_context']), 'cases': int(r['cases']), 'positives': int(r['positives']), 'share_positive_pct': round(float(r['share_positive_pct']), 4)})
inspection_detail = pd.DataFrame(inspection_detail)
save_csv(inspection_detail, '09_central_labels_by_inspection_context.csv', index=False)
inspection_detail


In [ ]:
# Eignung der Labels
prevalence_map = label_prevalence.set_index('label')['share_pct'].to_dict()
stab_pivot = stability_summary.pivot_table(index='label', columns='group', values='range_pp', aggfunc='first')
cramer_pivot = stability_summary.pivot_table(index='label', columns='group', values='cramers_v', aggfunc='first')
suitability_rows = []
for label in LABEL_COLS:
    prev = prevalence_map.get(label, np.nan)
    year_range = stab_pivot.loc[label, 'case_year'] if label in stab_pivot.index and 'case_year' in stab_pivot.columns else np.nan
    dept_range = stab_pivot.loc[label, 'case_department'] if label in stab_pivot.index and 'case_department' in stab_pivot.columns else np.nan
    inspection_range = stab_pivot.loc[label, 'inspection_context'] if label in stab_pivot.index and 'inspection_context' in stab_pivot.columns else np.nan
    jacc_primary = 1.0 if label == PRIMARY_LABEL else primary_overlap.set_index('label_b').get('jaccard', pd.Series(dtype=float)).get(label, np.nan)
    prevalence_score = 2 if 5 <= prev <= 20 else 1 if 3 <= prev < 5 or 20 < prev <= 30 else 0
    stability_score = 2 if pd.notna(year_range) and year_range <= 10 else 1 if pd.notna(year_range) and year_range <= 20 else 0
    inspection_score = 2 if pd.isna(inspection_range) or inspection_range <= 25 else 1 if inspection_range <= 50 else 0
    interpretability_score = 2
    if 'composite' in label.lower() or 'deptnorm' in label.lower():
        interpretability_score = 1
    if 'no_inspection' in label.lower():
        interpretability_score = 1
    total_score = prevalence_score + stability_score + inspection_score + interpretability_score
    if label == PRIMARY_LABEL:
        role_recommendation = 'Primärlabel'
    elif 'yearnorm' in label:
        role_recommendation = 'Robustheitslabel gegen Jahreseffekte'
    elif 'no_inspection' in label:
        role_recommendation = 'Robustheitslabel gegen Inspection-Dominanz'
    elif 'temporal' in label:
        role_recommendation = 'Sekundäre temporale Perspektive'
    elif 'change_or_objection' in label:
        role_recommendation = 'Vergleichslabel für Change und Objection'
    elif 'and' in label:
        role_recommendation = 'Strenger Robustheitscheck'
    else:
        role_recommendation = 'Sekundärlabel'
    suitability_rows.append({'label': label, 'prevalence_pct': prev, 'jaccard_with_primary': round(float(jacc_primary), 4) if pd.notna(jacc_primary) else np.nan, 'case_year_range_pp': round(float(year_range), 4) if pd.notna(year_range) else np.nan, 'department_range_pp': round(float(dept_range), 4) if pd.notna(dept_range) else np.nan, 'inspection_range_pp': round(float(inspection_range), 4) if pd.notna(inspection_range) else np.nan, 'prevalence_score_0_2': prevalence_score, 'year_stability_score_0_2': stability_score, 'inspection_score_0_2': inspection_score, 'interpretability_score_0_2': interpretability_score, 'total_heuristic_score_0_8': total_score, 'role_recommendation': role_recommendation})
suitability_matrix = pd.DataFrame(suitability_rows).sort_values('total_heuristic_score_0_8', ascending=False)
save_csv(suitability_matrix, '10_label_suitability_matrix.csv', index=False)
suitability_matrix


In [ ]:
# Merkmalsverfügbarkeit
def classify_feature_availability(feature):
    f = feature.lower()
    if f.startswith('label_'):
        return ('target_or_evaluation_only', 'Nicht als Feature nutzen; Label/Zielvariable oder Validierungslabel.', 'no')
    if f in ['event_count', 'duration_days', 'case_end', 'end_calendar_year', 'combined_rework_extra', 'repeated_combined_activity_total_extra', 'repeated_activity_total_extra']:
        return ('exclude_leakage_or_full_case_proxy', 'Vollständige Fallinformation; für frühe Prediction Leakage.', 'no')
    if any((k in f for k in ['payment_actual', 'penalty', 'rejected', 'success'])):
        return ('exclude_leakage_or_outcome', 'Outcome-/Nachprozess-/Audit-nahe Information; hohes Leakage-Risiko.', 'no')
    if any((k in f for k in ['identity:id', 'concept:name.1'])):
        return ('exclude_identifier_or_technical', 'Technische ID oder redundantes Konzeptfeld.', 'no')
    if any((k in f for k in ['applicant', 'case:concept:name'])):
        return ('exclude_or_historical_only', 'Identifier; nur für kontrollierte historische Features, nicht direkt als Modellfeature.', 'maybe_later')
    if any((k in f for k in ['selected_random', 'selected_risk', 'selected_manually'])):
        return ('requires_domain_check', 'Möglicherweise frühe Auswahl-/Kontrollinformation, aber Zeitpunkt muss fachlich geklärt werden.', 'maybe')
    if any((k in f for k in ['risk_factor', 'cross_compliance'])):
        return ('requires_domain_check', 'Potentiell früh verfügbare Risiko- oder Kontrollinformation; zeitliche Verfügbarkeit anhand der Datenbeschreibung prüfen.', 'maybe')
    if any((k in f for k in ['amount_applied', 'area', 'number_parcels', 'young farmer', 'small farmer', 'basic payment', 'greening', 'redistribution', 'program-id', 'department', 'case:year', 'application'])):
        return ('safe_static_candidate', 'Wahrscheinlich zu Prozessbeginn bzw. Antrag verfügbar; in Feature Availability dokumentieren.', 'yes_if_available_at_prediction_time')
    if f.startswith('act__') or f.startswith('subprocess__') or f.startswith('doctype__'):
        return ('exclude_full_case_event_aggregate', 'Full-case Event-Aggregat; für frühe Prediction nur als Prefix-Aggregat zulässig.', 'no_as_full_case')
    if any((k in f for k in ['case_start', 'start_calendar_year'])):
        return ('safe_time_context', 'Startzeit/Jahr kann als Kontextfeature genutzt werden; nicht mit Outcome verwechseln.', 'yes')
    return ('requires_review', 'Nicht automatisch eindeutig klassifizierbar; manuell prüfen.', 'maybe')
availability_rows = []
for col in case_df.columns:
    category, rationale, use_flag = classify_feature_availability(col)
    availability_rows.append({'feature': col, 'feature_group': 'case_attribute' if col.startswith('case:') else 'activity_count_full_case' if col.startswith('act__') else 'subprocess_count_full_case' if col.startswith('subprocess__') else 'doctype_count_full_case' if col.startswith('doctype__') else 'label' if col.startswith('label_') else 'case_level_metric', 'availability_class': category, 'use_for_early_prediction': use_flag, 'rationale': rationale})
prefix_feature_specs = [('prefix_event_count', 'prefix_dynamic_safe', 'Anzahl Events im Prefix; nur bis Prediction-Zeitpunkt.'), ('prefix_n_raw_activities', 'prefix_dynamic_safe', 'Anzahl unterschiedlicher Activities im Prefix.'), ('prefix_n_combined_activities', 'prefix_dynamic_safe', 'Kontextualisierte Aktivitätsvielfalt im Prefix.'), ('prefix_n_doctypes', 'prefix_dynamic_safe', 'Document-Type-Vielfalt im Prefix.'), ('prefix_n_subprocesses', 'prefix_dynamic_safe', 'Subprocess-Vielfalt im Prefix.'), ('prefix_n_resources', 'prefix_dynamic_safe', 'Resource-Vielfalt im Prefix.'), ('prefix_combined_rework_extra', 'prefix_dynamic_safe', 'Rework-Intensität nur innerhalb des Prefix.'), ('prefix_has_change_or_objection', 'prefix_dynamic_conditional', 'Nur zulässig, wenn Change/Objection bis zum Prediction-Zeitpunkt bereits beobachtet wurde.'), ('prefix_has_inspection_context', 'prefix_dynamic_conditional', 'Nur zulässig, wenn Inspection-Kontext bis zum Prediction-Zeitpunkt beobachtet wurde; kann fachlich stark sein.')]
for feat, avail, rationale in prefix_feature_specs:
    availability_rows.append({'feature': feat, 'feature_group': 'prefix_dynamic_feature', 'availability_class': avail, 'use_for_early_prediction': 'yes_if_prefix_based', 'rationale': rationale})
feature_availability = pd.DataFrame(availability_rows)
save_csv(feature_availability, '11_feature_availability_matrix.csv', index=False)
feature_availability_summary = feature_availability['availability_class'].value_counts().rename_axis('availability_class').reset_index(name='feature_count')
save_csv(feature_availability_summary, '12_feature_availability_summary.csv', index=False)
feature_availability_summary


In [ ]:
# Präfixmerkmale
def aggregate_prefix_features(prefix_events, prefix_name, require_min_events=None):
    if len(prefix_events) == 0:
        return pd.DataFrame()
    agg_dict = {'prefix_event_count': (ACTIVITY_COL, 'size'), 'prefix_n_raw_activities': (ACTIVITY_COL, 'nunique'), 'prefix_n_combined_activities': ('combined_activity', 'nunique')}
    if 'doctype' in prefix_events.columns:
        agg_dict['prefix_n_doctypes'] = ('doctype', 'nunique')
    if 'subprocess' in prefix_events.columns:
        agg_dict['prefix_n_subprocesses'] = ('subprocess', 'nunique')
    if 'org:resource' in prefix_events.columns:
        agg_dict['prefix_n_resources'] = ('org:resource', 'nunique')
    pref = prefix_events.groupby(CASE_COL).agg(**agg_dict).reset_index()
    cc = prefix_events.groupby([CASE_COL, 'combined_activity']).size().reset_index(name='count')
    cc_rep = cc[cc['count'] > 1]
    if len(cc_rep) > 0:
        pref_rework = cc_rep.groupby(CASE_COL).agg(prefix_combined_rework_extra=('count', lambda x: int((x - 1).sum())), prefix_repeated_combined_activity_types=('combined_activity', 'nunique')).reset_index()
        pref = pref.merge(pref_rework, on=CASE_COL, how='left')
    else:
        pref['prefix_combined_rework_extra'] = 0
        pref['prefix_repeated_combined_activity_types'] = 0
    for c in ['prefix_combined_rework_extra', 'prefix_repeated_combined_activity_types']:
        pref[c] = pd.to_numeric(pref.get(c, 0), errors='coerce').fillna(0)
    if 'subprocess' in prefix_events.columns:
        co_flag = prefix_events['subprocess'].astype(str).str.contains('change|objection', case=False, regex=True, na=False)
        co = co_flag.groupby(prefix_events[CASE_COL]).max().rename('prefix_has_change_or_objection').reset_index()
        pref = pref.merge(co, on=CASE_COL, how='left')
    else:
        pref['prefix_has_change_or_objection'] = False
    ins = prefix_events.groupby(CASE_COL)['is_inspection_context'].max().rename('prefix_has_inspection_context').reset_index()
    pref = pref.merge(ins, on=CASE_COL, how='left')
    pref['prefix_has_inspection_context'] = pref['prefix_has_inspection_context'].fillna(False).astype(bool)
    pref['prefix_strategy'] = prefix_name
    if require_min_events is not None:
        pref['prefix_requires_min_events'] = require_min_events
    return pref
prefix_matrices = []
prefix_feasibility_rows = []
if BUILD_PREFIX_FEATURES:
    label_attach_cols = [CASE_COL, PRIMARY_LABEL] + [c for c in ['label_scd_p90_or_yearnorm', 'label_scd_p90_or_no_inspection_global', 'label_path_change_or_objection', 'label_temporal_duration_p90_global'] if c in case_df.columns]
    label_attach = case_df[label_attach_cols].copy()
    for k in PREFIX_K_LIST:
        eligible_cases = case_df.loc[case_df['event_count'] >= k, CASE_COL]
        pref_events = event_df[event_df['event_position'] <= k].copy()
        pref = aggregate_prefix_features(pref_events, f'first_{k}_events', require_min_events=k)
        pref = pref.merge(label_attach, on=CASE_COL, how='left')
        pref['eligible_for_strategy'] = pref[CASE_COL].isin(set(eligible_cases))
        prefix_matrices.append(pref)
        eligible_pref = pref[pref['eligible_for_strategy']].copy()
        prefix_feasibility_rows.append({'prefix_strategy': f'first_{k}_events', 'cases_available': int(len(eligible_pref)), 'case_share_pct': round(float(len(eligible_pref) / len(case_df) * 100), 4), 'positive_cases_primary': int(eligible_pref[PRIMARY_LABEL].fillna(False).astype(bool).sum()), 'positive_share_primary_pct': round(float(eligible_pref[PRIMARY_LABEL].fillna(False).astype(bool).mean() * 100), 4) if len(eligible_pref) else np.nan, 'median_prefix_event_count': float(eligible_pref['prefix_event_count'].median()) if len(eligible_pref) else np.nan})
    for days in PREFIX_DAY_LIST:
        pref_events = event_df[event_df['days_since_case_start'] <= days].copy()
        pref = aggregate_prefix_features(pref_events, f'first_{days}_days', require_min_events=None)
        pref = pref.merge(label_attach, on=CASE_COL, how='right')
        for c in ['prefix_event_count', 'prefix_n_raw_activities', 'prefix_n_combined_activities', 'prefix_n_doctypes', 'prefix_n_subprocesses', 'prefix_n_resources', 'prefix_combined_rework_extra']:
            if c in pref.columns:
                pref[c] = pd.to_numeric(pref[c], errors='coerce').fillna(0)
        for c in ['prefix_has_change_or_objection', 'prefix_has_inspection_context']:
            if c in pref.columns:
                pref[c] = pref[c].fillna(False).astype(bool)
        pref['prefix_strategy'] = f'first_{days}_days'
        pref['eligible_for_strategy'] = True
        prefix_matrices.append(pref)
        prefix_feasibility_rows.append({'prefix_strategy': f'first_{days}_days', 'cases_available': int(len(pref)), 'case_share_pct': round(float(len(pref) / len(case_df) * 100), 4), 'positive_cases_primary': int(pref[PRIMARY_LABEL].fillna(False).astype(bool).sum()), 'positive_share_primary_pct': round(float(pref[PRIMARY_LABEL].fillna(False).astype(bool).mean() * 100), 4) if len(pref) else np.nan, 'median_prefix_event_count': float(pref['prefix_event_count'].median()) if 'prefix_event_count' in pref.columns else np.nan})
prefix_feasibility = pd.DataFrame(prefix_feasibility_rows)
save_csv(prefix_feasibility, '13_prefix_feasibility_summary.csv', index=False)
if prefix_matrices:
    prefix_feature_matrix = pd.concat(prefix_matrices, ignore_index=True)
    save_csv(prefix_feature_matrix, '14_prefix_feature_matrix_core.csv', index=False)
else:
    prefix_feature_matrix = pd.DataFrame()
prefix_feasibility


In [ ]:
# Abbildungen
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = label_prevalence.sort_values('share_pct', ascending=True)
ax.barh(plot_df['label'], plot_df['share_pct'])
ax.set_xlabel('Anteil Fälle (%)')
ax.set_title('Prävalenz der Robustheitslabels')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_01_label_prevalence_robustness.png')
if len(primary_overlap):
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_df = primary_overlap.sort_values('jaccard', ascending=True)
    ax.barh(plot_df['label_b'], plot_df['jaccard'])
    ax.set_xlabel('Jaccard-Overlap mit Primary Label')
    ax.set_title('Overlap der Robustheitslabels mit SCD-P90-OR')
    ax.grid(axis='x', alpha=0.3)
    save_fig(fig, 'fig_02_jaccard_with_primary.png')
if len(year_detail):
    pivot = year_detail.pivot(index=CASE_YEAR_COL, columns='label', values='share_positive_pct')
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, aspect='auto')
    ax.set_xticks(np.arange(pivot.shape[1]))
    ax.set_yticks(np.arange(pivot.shape[0]))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticklabels(pivot.index)
    ax.set_title('Label-Prävalenz nach case:year')
    fig.colorbar(im, ax=ax, label='positive Fälle (%)')
    save_fig(fig, 'fig_03_label_by_case_year_heatmap.png')
if 'label_scd_p90_or_no_inspection_global' in case_df.columns:
    comp = pd.DataFrame([pair_overlap_metrics(case_df, PRIMARY_LABEL, 'label_scd_p90_or_no_inspection_global')])
    values = [comp.loc[0, 'both_true'], comp.loc[0, 'only_a'], comp.loc[0, 'only_b']]
    labels = ['beide', 'nur Primary', 'nur No-Inspection']
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(labels, values)
    ax.set_ylabel('Anzahl Cases')
    ax.set_title('Primary SCD vs. No-Inspection SCD')
    ax.grid(axis='y', alpha=0.3)
    save_fig(fig, 'fig_04_primary_vs_no_inspection.png')
fig, ax = plt.subplots(figsize=(9, 5))
fas = feature_availability_summary.sort_values('feature_count', ascending=True)
ax.barh(fas['availability_class'], fas['feature_count'])
ax.set_xlabel('Anzahl Features')
ax.set_title('Feature Availability Klassen')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_05_feature_availability_summary.png')
if len(prefix_feasibility):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(prefix_feasibility['prefix_strategy'], prefix_feasibility['positive_cases_primary'])
    ax.set_ylabel('Positive Cases Primary Label')
    ax.set_title('Positive Fälle je Prefix-Strategie')
    ax.set_xticklabels(prefix_feasibility['prefix_strategy'], rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    save_fig(fig, 'fig_06_prefix_positive_cases.png')
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = suitability_matrix.sort_values('total_heuristic_score_0_8', ascending=True)
ax.barh(plot_df['label'], plot_df['total_heuristic_score_0_8'])
ax.set_xlabel('heuristischer Eignungsscore (0–8)')
ax.set_title('Eignung der Labelvarianten')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_07_label_suitability_score.png')
print('Abbildungen erstellt:', len(created_figures))


In [ ]:
# Falldaten speichern
selected_case_cols = [CASE_COL, 'event_count', 'duration_days', 'combined_rework_extra', 'event_count_no_inspection', 'combined_rework_extra_no_inspection', 'has_inspection_context', 'inspection_event_count', CASE_YEAR_COL]
if CASE_DEPT_COL is not None and CASE_DEPT_COL in case_df.columns:
    selected_case_cols.append(CASE_DEPT_COL)
selected_case_cols += [c for c in LABEL_COLS if c in case_df.columns]
selected_case_cols = [c for c in selected_case_cols if c in case_df.columns]
case_export = case_df[selected_case_cols].copy()
save_csv(case_export, '17_case_level_label_robustness_core.csv', index=False)
